# Quality Control and Read Alignment with FastQC, fastp, MultiQC and BWA

**Course:** Population Genomics and Population Structure Analysis  
**Instructor:** Nikolay Oskolkov  
**Platform:** Google Colab

---

In this notebook, we will:

1. Install **Conda**, **FastQC**, **MultiQC**, **fastp**, **BWA**, **SAMtools**, and **BEDTools** in Google Colab.
2. Mount **Google Drive** to access the gzipped FASTQ files.
3. Run **FastQC** on the raw reads and aggregate the reports with **MultiQC** (folder: `QC_before_adapter_removal`).
4. Perform **adapter trimming** with **fastp**, writing trimmed reads to `trimmed_fastqs`.
5. Run **FastQC** again on the trimmed reads and aggregate with **MultiQC** (folder: `QC_after_adapter_removal`).
6. Download the human reference genome **hg38** from UCSC into `hg38` and index it with **BWA**.
7. Align every trimmed FASTQ file to the reference with **bwa mem** in a loop, writing sorted BAM files to `bams`.

**Input data:** gzipped FASTQ files (`*.fastq.gz`) from the 1000 Genomes Project, located in your Google Drive at `PopGen_PopStructure/smallfastqs/`.

## 1. Install Conda in Google Colab

Google Colab does not ship with Conda by default. We use the `condacolab` package, which installs a lightweight Miniconda environment directly inside the notebook.

> ⚠️ **Important:** After running the cell below, the Colab kernel will **restart automatically**. This is expected — just continue with the next cell once the restart finishes.

In [1]:
# Install Conda inside Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:13
🔁 Restarting kernel...


## 2. Install FastQC, MultiQC, fastp, BWA, SAMtools, and BEDTools

All tools are available from **Bioconda**. We install them into the base Conda environment so they are immediately available on the PATH.

| Tool | Purpose |
|---|---|
| **FastQC** | Per-sample read quality control |
| **MultiQC** | Aggregate FastQC reports into a single HTML page |
| **fastp** | Adapter and quality trimming |
| **BWA** | Read alignment to a reference genome |
| **SAMtools** | Sort and index BAM files |
| **BEDTools** | Optional: BAM/interval manipulation downstream |

The installation may take a couple of minutes.

In [1]:
# Install bioinformatics tools from Bioconda
!conda install -y bioconda::fastqc
!conda install -y bioconda::multiqc
!conda install -y bioconda::fastp
!conda install -y bioconda::bwa
!conda install -y bioconda::samtools
!conda install -y bioconda::bedtools

Retrieving notices: - \ done
Channels:
 - conda-forge
 - bioconda
Platform: linux-64
Solving environment: / - \ | / done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - bioconda::fastqc


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _python_abi3_support-1.0   |       hd8ed1ab_3           8 KB  conda-forge
    alsa-lib-1.2.16.1          |       h7cc23a3_1         581 KB  conda-forge
    annotated-types-0.8.0      |     pyhd8ed1ab_0          19 KB  conda-forge
    anyio-4.15.1               |     pyh5ded981_1         171 KB  conda-forge
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    cached-property-2.0.1      |     pyhcf101f3_0           4 KB  conda-forge
    cached_property-2.0.1      |     pyhcf101f3_0          16 KB  conda-forge
    cairo-1.18.4               |       he90730b_1         966 KB  co

### 2.1 Verify the installation

Let us confirm that all tools are available and print their versions.

In [2]:
!fastqc --version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
FastQC v0.12.1


In [3]:
!multiqc --version

multiqc, version 1.35


In [4]:
!fastp --version

fastp 1.3.7


In [5]:
!bwa 2>&1 | head -n 5


Program: bwa (alignment via Burrows-Wheeler transformation)
Version: 0.7.19-r1273
Contact: Heng Li <hli@ds.dfci.harvard.edu>



In [6]:
!samtools --version | head -n 2

samtools 1.23.1
Using htslib 1.23.1


In [7]:
!bedtools --version

bedtools v2.31.1


## 3. Mount Google Drive

We will store the input FASTQ files, the trimmed reads, the reference genome, and the BAM files on Google Drive, so that all results persist between Colab sessions.

Running the cell below will open a Google authorization dialog. Grant access to your Drive, then return to the notebook.

In [8]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Set the Working Directory

We change the working directory to the project folder on Google Drive. All output folders (`QC_before_adapter_removal`, `trimmed_fastqs`, `QC_after_adapter_removal`, `hg38`, `bams`) will be created relative to this directory.

In [9]:
import os
# Change working directory to the 1000G project folder on Google Drive
os.chdir("/content/drive/MyDrive/PopGen_PopStructure/1000G")

In [10]:
# Confirm we are in the right place
!pwd
!ls -lh

/content/drive/MyDrive/PopGen_PopStructure/1000G
total 4.2M
-rw------- 1 root root 8.3K Sep 27 14:35 1000G.arg
-rw------- 1 root root  53K Sep 27 13:39 1000G_bam_list.txt
-rw------- 1 root root 2.2M Sep 27 14:35 1000G.beagle.gz
-rw------- 1 root root  42K Sep 27 14:47 1000G.fopt.gz
-rw------- 1 root root  471 Sep 27 14:47 1000G.log
-rw------- 1 root root  13K Sep 27 14:35 1000G.mafs.gz
-rw------- 1 root root  30K Sep 27 14:47 1000G.qopt
drwx------ 2 root root 4.0K Sep 27 05:23 beagle
-rw------- 1 root root 2.0M Sep 27 14:35 pca_1000G.cov
-rw------- 1 root root  369 Sep 27 14:35 pca_1000G.log
drwx------ 2 root root 4.0K Sep 27 05:23 smallbams
drwx------ 2 root root 4.0K Sep 28 10:05 smallfastqs


## 5. Check the Input FASTQ Files

The gzipped FASTQ files should already be present in the `smallfastqs/` folder. Let us confirm the number of files and inspect a couple of them.

In [11]:
!ls smallfastqs/*.fastq.gz | wc -l
!ls -lh smallfastqs/*.fastq.gz | head

435
-rw------- 1 root root 392K Sep 28 09:56 smallfastqs/small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 1.2M Sep 28 09:56 smallfastqs/small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 909K Sep 28 09:56 smallfastqs/small.NA06986.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415.fastq.gz
-rw------- 1 root root 256K Sep 28 09:56 smallfastqs/small.NA06989.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 375K Sep 28 09:56 smallfastqs/small.NA06994.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 743K Sep 28 09:56 smallfastqs/small.NA07000.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415.fastq.gz
-rw------- 1 root root 528K Sep 28 09:56 smallfastqs/small.NA07037.mapped.ILLUMINA.bwa.CEU.low_coverage.20130502.fastq.gz
-rw------- 1 root root 282K Sep 28 09:56 smallfastqs/small.NA07048.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root ro

## 6. Create Output Folders

We create the output folder structure up front, so that every subsequent step simply writes into an existing directory.

| Folder | Content |
|---|---|
| `QC_before_adapter_removal/` | FastQC and MultiQC reports on raw reads |
| `trimmed_fastqs/` | Adapter- and quality-trimmed FASTQ files |
| `QC_after_adapter_removal/` | FastQC and MultiQC reports on trimmed reads |
| `hg38/` | Human reference genome and BWA index |
| `bams/` | Sorted BAM alignments |

In [13]:
!mkdir -p QC_before_adapter_removal
!mkdir -p trimmed_fastqs
!mkdir -p QC_after_adapter_removal
!mkdir -p hg38
!mkdir -p bams

!ls -d */

bams/	 hg38/			    QC_before_adapter_removal/	smallfastqs/
beagle/  QC_after_adapter_removal/  smallbams/			trimmed_fastqs/


## 7. Quality Control on Raw Reads with FastQC

**FastQC** produces per-sample HTML and ZIP reports that summarise:

- Per-base quality scores
- Per-sequence quality scores
- GC content
- Adapter content
- Overrepresented sequences
- Duplication levels

We run FastQC on every `.fastq.gz` file in `smallfastqs/` and write the reports to `QC_before_adapter_removal/`.

For a real dataset, FastQC can be slow if run serially. The loop below processes files one at a time. If you have many files and want to parallelise, replace the loop with `ls smallfastqs/*.fastq.gz | xargs -P 4 -I {} fastqc -o QC_before_adapter_removal {}`.

In [14]:
%%bash
for fq in smallfastqs/*.fastq.gz; do
    echo "Running FastQC on $fq ..."
    fastqc -o QC_before_adapter_removal "$fq"
done

echo "Done. FastQC reports written to QC_before_adapter_removal/"

Running FastQC on smallfastqs/small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz ...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
Analysis complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Running FastQC on smallfastqs/small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz ...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
Analysis co

Started analysis of small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 15% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 30% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 45% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 65% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 80% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 95% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Started analysis of small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 5% complete for small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 10% complete for small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 20% complete for small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.2012

### 7.1 Aggregate FastQC Reports with MultiQC

**MultiQC** scans a directory, finds all FastQC reports, and combines them into a single interactive HTML page. This makes it easy to spot batch effects, samples with poor quality, or systematic adapter contamination.

We run MultiQC on `QC_before_adapter_removal/` and write the aggregated report back into the same folder.

In [15]:
%%bash

multiqc QC_before_adapter_removal -o QC_before_adapter_removal -n multiqc_before_adapter_removal

!ls -lh QC_before_adapter_removal/


/// MultiQC 🔍 v1.35

       file_search | Search path: /content/drive/MyDrive/PopGen_PopStructure/1000G/QC_before_adapter_removal
            fastqc | Found 435 reports
     write_results | Data        : QC_before_adapter_removal/multiqc_before_adapter_removal_data
     write_results | Report      : QC_before_adapter_removal/multiqc_before_adapter_removal.html
           multiqc | MultiQC complete
bash: line 4: !ls: command not found


CalledProcessError: Command 'b'\nmultiqc QC_before_adapter_removal -o QC_before_adapter_removal -n multiqc_before_adapter_removal\n\n!ls -lh QC_before_adapter_removal/\n'' returned non-zero exit status 127.

### 7.2 Inspect the MultiQC Report

Open `QC_before_adapter_removal/multiqc_before_adapter_removal.html` in a browser to review:

- **Per Base Sequence Quality** — do quality scores drop towards the 3′ end?
- **Adapter Content** — is adapter contamination present?
- **Per Sequence GC Content** — is there unusual GC bias?
- **Sequence Duplication Levels** — are libraries over-amplified?

If adapter content is significant, trimming with fastp is strongly recommended before alignment.

## 8. Adapter and Quality Trimming with fastp

**fastp** performs automatic adapter detection and removal, quality filtering, and produces its own HTML/JSON reports. It is faster and more accurate than older tools such as Trimmomatic for typical Illumina data.

The loop below runs fastp on every `.fastq.gz` file and writes trimmed reads to `trimmed_fastqs/` with the same filename. Per-sample HTML and JSON reports are also written there for inspection.

Common fastp options used here:

| Option | Meaning |
|---|---|
| `-i` | Input FASTQ (gzipped) |
| `-o` | Output FASTQ (gzipped) |
| `-w 4` | Use 4 threads per sample |
| `-q 20` | Minimum base quality |
| `-u 30` | Max fraction of low-quality bases per read |
| `-l 50` | Minimum read length after trimming |
| `-j` | JSON report path |
| `-h` | HTML report path |

The adapter sequences are detected automatically; you do not need to specify them.

In [16]:
%%bash

mkdir -p trimmed_fastqs/fastp_reports

for fq in smallfastqs/*.fastq.gz; do
    sample=$(basename "$fq" .fastq.gz)
    echo "Trimming $sample ..."
    fastp \
        -i "$fq" \
        -o "trimmed_fastqs/${sample}.fastq.gz" \
        -w 4 \
        -q 20 \
        -u 30 \
        -l 50 \
        -j "trimmed_fastqs/fastp_reports/${sample}.json" \
        -h "trimmed_fastqs/fastp_reports/${sample}.html"
done

echo "Done. Trimmed reads written to trimmed_fastqs/"

Trimming small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522 ...
Trimming small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522 ...
Trimming small.NA06986.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415 ...
Trimming small.NA06989.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522 ...
Trimming small.NA06994.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522 ...
Trimming small.NA07000.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415 ...
Trimming small.NA07037.mapped.ILLUMINA.bwa.CEU.low_coverage.20130502 ...
Trimming small.NA07048.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522 ...
Trimming small.NA07051.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522 ...
Trimming small.NA07056.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415 ...
Trimming small.NA07347.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415 ...
Trimming small.NA07357.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415 ...
Trimming small.NA10847.mapped.ILLUMINA.bwa.CEU.low_coverage.20130502 ...
Trimming small.NA10851.mapped.ILLUMINA.bwa.CEU.low_

Detecting adapter sequence for read1...
No adapter detected for read1

Read1 before filtering:
total reads: 6158
total bases: 468008
Q20 bases: 425974(91.0185%)
Q30 bases: 396038(84.6221%)
Q40 bases: 192043(41.0341%)

Read1 after filtering:
total reads: 5600
total bases: 425600
Q20 bases: 409743(96.2742%)
Q30 bases: 384197(90.2719%)
Q40 bases: 188925(44.3903%)

Filtering result:
reads passed filter: 5600
reads failed due to low quality: 557
reads failed due to too many N: 1
reads failed due to too short: 0
reads failed due to adapter dimer: 0
reads with adapter trimmed: 0
bases trimmed due to adapters: 0

Duplication rate (may be overestimated since this is SE data): 1.6239%

JSON report: trimmed_fastqs/fastp_reports/small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.json
HTML report: trimmed_fastqs/fastp_reports/small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.html

fastp -i smallfastqs/small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz -o trimmed

### 8.1 Check the Trimmed FASTQ Files

We should see the same number of `.fastq.gz` files in `trimmed_fastqs/` as in `smallfastqs/`.

In [20]:
%%bash

ls trimmed_fastqs/*.fastq.gz | wc -l
ls -lh trimmed_fastqs/*.fastq.gz | head

435
-rw------- 1 root root 365K Sep 28 10:53 trimmed_fastqs/small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 1.1M Sep 28 10:53 trimmed_fastqs/small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 815K Sep 28 10:53 trimmed_fastqs/small.NA06986.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415.fastq.gz
-rw------- 1 root root 246K Sep 28 10:53 trimmed_fastqs/small.NA06989.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 365K Sep 28 10:53 trimmed_fastqs/small.NA06994.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
-rw------- 1 root root 711K Sep 28 10:53 trimmed_fastqs/small.NA07000.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415.fastq.gz
-rw------- 1 root root 510K Sep 28 10:53 trimmed_fastqs/small.NA07037.mapped.ILLUMINA.bwa.CEU.low_coverage.20130502.fastq.gz
-rw------- 1 root root 262K Sep 28 10:53 trimmed_fastqs/small.NA07048.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq

## 9. Quality Control on Trimmed Reads with FastQC

After adapter removal, we re-run FastQC on the trimmed reads to confirm that adapter content has been removed and that quality scores are acceptable.

Reports are written to `QC_after_adapter_removal/`.

In [18]:
%%bash

for fq in trimmed_fastqs/*.fastq.gz; do
    echo "Running FastQC on $fq ..."
    fastqc -o QC_after_adapter_removal "$fq"
done

echo "Done. FastQC reports written to QC_after_adapter_removal/"

Running FastQC on trimmed_fastqs/small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz ...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
Analysis complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Running FastQC on trimmed_fastqs/small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz ...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
Analy

Started analysis of small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 15% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 35% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 50% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 70% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 85% complete for small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Started analysis of small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 5% complete for small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 15% complete for small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 20% complete for small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522.fastq.gz
Approx 30% complete for small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.2012

### 9.1 Aggregate FastQC Reports with MultiQC

We run MultiQC again on the post-trimming reports.

In [21]:
%%bash

multiqc QC_after_adapter_removal -o QC_after_adapter_removal -n multiqc_after_adapter_removal

ls -lh QC_after_adapter_removal/

total 407M
-rw------- 1 root root 8.3M Sep 28 11:32 multiqc_after_adapter_removal_1.html
drwx------ 2 root root 4.0K Sep 28 11:31 multiqc_after_adapter_removal_data
drwx------ 2 root root 4.0K Sep 28 11:32 multiqc_after_adapter_removal_data_1
-rw------- 1 root root 8.2M Sep 28 11:31 multiqc_after_adapter_removal.html
-rw------- 1 root root 558K Sep 28 11:01 small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522_fastqc.html
-rw------- 1 root root 336K Sep 28 11:01 small.NA06984.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522_fastqc.zip
-rw------- 1 root root 588K Sep 28 11:01 small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522_fastqc.html
-rw------- 1 root root 339K Sep 28 11:01 small.NA06985.mapped.ILLUMINA.bwa.CEU.low_coverage.20120522_fastqc.zip
-rw------- 1 root root 593K Sep 28 11:01 small.NA06986.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415_fastqc.html
-rw------- 1 root root 343K Sep 28 11:01 small.NA06986.mapped.ILLUMINA.bwa.CEU.low_coverage.20130415_fastqc.zip
-rw---


/// MultiQC 🔍 v1.35

       file_search | Search path: /content/drive/MyDrive/PopGen_PopStructure/1000G/QC_after_adapter_removal
            fastqc | Found 435 reports
      exec_modules | Updating module "FastQC"
     write_results | Existing reports found, adding suffix to filenames. Use '--force' to overwrite.
     write_results | Data        : QC_after_adapter_removal/multiqc_after_adapter_removal_data_1
     write_results | Report      : QC_after_adapter_removal/multiqc_after_adapter_removal_1.html
           multiqc | MultiQC complete


## 10. Download the Human Reference Genome hg38

We download the human reference genome **hg38** from the UCSC goldenPath server. The file `hg38.fa.gz` is approximately 1 GB and will be placed in the `hg38/` folder.

The reference is a plain FASTA file. After download, we decompress it once and leave the uncompressed `.fa` file in place, because BWA and downstream tools prefer random access to an uncompressed FASTA.

In [22]:
%%bash

cd hg38

# Download the reference genome (skip if already present)
[ -f hg38.fa.gz ] || wget http://hgdownload.soe.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz

# Decompress (skip if already decompressed)
[ -f hg38.fa ] || gunzip -k hg38.fa.gz

ls -lh

cd ..

total 4.0G
-rw------- 1 root root 3.1G Jan 16  2014 hg38.fa
-rw------- 1 root root 939M Jan 16  2014 hg38.fa.gz


--2026-09-28 11:32:42--  http://hgdownload.soe.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 983659424 (938M) [application/x-gzip]
Saving to: ‘hg38.fa.gz’

     0K .......... .......... .......... .......... ..........  0%  480K 33m19s
    50K .......... .......... .......... .......... ..........  0%  918K 25m23s
   100K .......... .......... .......... .......... ..........  0% 18.8M 17m12s
   150K .......... .......... .......... .......... ..........  0% 60.5M 12m58s
   200K .......... .......... .......... .......... ..........  0%  966K 13m41s
   250K .......... .......... .......... .......... ..........  0% 95.0M 11m26s
   300K .......... .......... .......... .......... ..........  0% 28.2M 9m53s
   350K .......... .......... .......... .......... ...

### 10.1 Index the Reference Genome with BWA

**BWA** requires its own index, which is stored in five files with extensions:

- `.amb` — ambiguity
- `.ann` — annotation
- `.bwt` — Burrows–Wheeler transform
- `.pac` — packed reference
- `.sa` — suffix array

Indexing the human genome takes a few minutes and requires several GB of RAM. The index is written next to `hg38.fa` in the `hg38/` folder.

After BWA, we also create a **FASTA index** (`.fai`) and a **sequence dictionary** for SAMtools, which are needed downstream.

In [ ]:
%%bash

cd hg38

# BWA index (skip if already present)
[ -f hg38.fa.bwt ] || bwa index hg38.fa

# SAMtools FASTA index
[ -f hg38.fa.fai ] || samtools faidx hg38.fa

ls -lh

cd ..

## 11. Align Trimmed Reads to hg38 with bwa mem

We now align every trimmed FASTQ file to the reference genome with **bwa mem**, producing a SAM file that we immediately convert to a **sorted BAM** with **SAMtools**.

Key parameters:

| Parameter | Meaning |
|---|---|
| `-t 4` | Use 4 threads for alignment |
| `-R '@RG\tID:...\tSM:...'` | Add a read group with sample name, required by GATK and ANGSD |
| `samtools sort` | Sort BAM by coordinate for downstream tools |
| `samtools index` | Index the sorted BAM (`.bai`) |

The final BAM files are written to `bams/<sample>.sorted.bam`. Intermediate SAM files are deleted to save space.

In [ ]:
%%bash

REF=hg38/hg38.fa

for fq in trimmed_fastqs/*.fastq.gz; do
    sample=$(basename "$fq" .fastq.gz)
    echo "Aligning $sample ..."

    bwa mem -t 4 \
        -R "@RG\tID:${sample}\tSM:${sample}\tPL:ILLUMINA\tLB:lib1\tPU:unit1" \
        "$REF" "$fq" \
    | samtools sort -@ 4 -o "bams/${sample}.sorted.bam" -

    samtools index "bams/${sample}.sorted.bam"
done

echo "Done. Sorted BAM files written to bams/"

### 11.1 Inspect the BAM Files

We can quickly check a BAM file with `samtools flagstat` to see how many reads aligned, how many were properly paired, and how many were duplicates.

In [ ]:
%%bash

ls -lh bams/*.sorted.bam | head

# Example: flagstat on the first BAM
first_bam=$(ls bams/*.sorted.bam | head -n 1)
echo "=== Flagstat for $first_bam ==="
samtools flagstat "$first_bam"

### 11.2 Optional: Aggregate Alignment Statistics with MultiQC

MultiQC can also aggregate **SAMtools stats** and **Picard** metrics. To do so, first generate `samtools stats` for each BAM:

```bash
mkdir -p bams/samtools_stats
for bam in bams/*.sorted.bam; do
    sample=$(basename "$bam" .sorted.bam)
    samtools stats "$bam" > "bams/samtools_stats/${sample}.stats"
done
multiqc bams/samtools_stats -o bams -n multiqc_alignment

Open bams/multiqc_alignment.html to review alignment metrics across all samples.


---

## Cell 38 — Markdown (Summary)

```markdown
## 12. Summary

In this notebook we:

✅ Installed **Conda**, **FastQC**, **MultiQC**, **fastp**, **BWA**, **SAMtools**, and **BEDTools** in Google Colab.  
✅ Mounted **Google Drive** to access the gzipped FASTQ files.  
✅ Ran **FastQC** on raw reads and aggregated with **MultiQC** (`QC_before_adapter_removal/`).  
✅ Performed **adapter and quality trimming** with **fastp** (`trimmed_fastqs/`).  
✅ Ran **FastQC** on trimmed reads and aggregated with **MultiQC** (`QC_after_adapter_removal/`).  
✅ Downloaded and indexed the human reference genome **hg38** (`hg38/`).  
✅ Aligned every trimmed FASTQ to hg38 with **bwa mem**, producing sorted BAMs (`bams/`).

### Next steps

- Run **ANGSD** on the sorted BAM files to estimate genotype likelihoods from low-coverage data.
- Run **PCA** with **pcangsd** and **admixture** with **NGSadmix** on the resulting Beagle file.
- Compute **Fst**, **F2**, **F3**, and **D-statistics** with **ANGSD** or **admixtools**.

### References

- Andrews, S. (2010). *FastQC: A Quality Control Tool for High Throughput Sequence Data.*  
- Ewels, P., et al. (2016). *MultiQC: Summarize analysis results for multiple tools and samples in a single report.* Bioinformatics.  
- Chen, S., et al. (2018). *fastp: an ultra-fast all-in-one FASTQ preprocessor.* Bioinformatics.  
- Li, H., & Durbin, R. (2009). *Fast and accurate short read alignment with Burrows–Wheeler transform.* Bioinformatics.  
- Li, H., et al. (2009). *The Sequence Alignment/Map format and SAMtools.* Bioinformatics.

In [ ]:
!jupyter nbconvert --to html /content/drive/MyDrive/PopGen_PopStructure/QC_Alignment_Colab.ipynb